Here we check how the network influences the behaviour of the automaton. We load all networks that are also used by Lucas in the LLNA papers.

# General remarks

- The behaviour depends on the network. It is _not_ the case that rule $\phi^9_{488,464}$ always leads to a 'totalitarian' outcome.
- We should figure out whether this particular rule is good at network identification
- We should experiment with other self-equivalent rules that qualify, and see whether some of these _do_ lead to a homogeneous outcome.

In [ ]:
# standard preamble for the Notebooks I use
import torch as tc
import igraph as ig
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import rcParams

# Enable LaTeX and set Times New Roman as the font
rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "text.latex.preamble": r"\usepackage{amsmath}"  # Optional: Use LaTeX packages
})

from tqdm import tqdm

import sys, os
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.append(parent_dir)

# compact saving of data
import h5py

from src.automata import LLNA
from src.simulation import *
from src.rules import binary_indices, return_equivalent_rule, get_nonequiv_rules, return_life_like_dict
from src.networks import create_2d_torus_lattice, watts_strogatz_rewire
from src.analysis import median_and_percentiles_over_ensemble, mean_field_dens_propagation, derrida_map_analytical, hamming_weight, boolean_sens, mean_field_slope

%load_ext autoreload
%autoreload 2

In [ ]:
# load additional classes
import numpy as np
import torch as tc
import torch_geometric as tg
import torch.nn.functional as F

# from torch_sparse import SparseTensor
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as GraphDataLoader
from torch.utils.data import Dataset, DataLoader, Subset, RandomSampler, WeightedRandomSampler
from sklearn.model_selection import train_test_split

from typing import Union

#===========================================================
class DiscreteStateNetwork(Data):
    def __init__(self, edge_index:list=[[0], [0]], y:int=0, num_states:int=2, num_inits:int=1, *args, **kwargs):
        """
        `edge_index`: indices of the edges (i,j) in the form [[i1, ... im], [j1, ... jm]]
        `y`: integer code of the respective class
        `num_states`: the number of states to randomly generate the state vector (default: binary network)
        `num_inits`: the number of different initial configurations to generate, also affects the y vector (expansion)
        """
        num_nodes = np.max(edge_index) + 1
        super(DiscreteStateNetwork, self).__init__(
            edge_index = tc.tensor(edge_index, dtype=tc.long), # shape of [2, |E|]
            node_state = tc.randint(0, num_states, (num_nodes, num_inits)).float(), # shape of [|V|, L]
            y = tc.tensor(y).expand(num_inits), # shape of [L]
            batch = tc.zeros(num_nodes), # shape of [|V|]
            *args, **kwargs
        )

    def unpack(self):
        return tuple([self.edge_index, self.node_state.T, self.batch, self.y])


#===========================================================
class DiscreteStateNetworkDataset(Dataset):
    def __init__(self, network_collection, device:str='cpu', *args, **kwargs):
        classes, labels = np.unique(network_collection['label'], return_inverse=True)
        for G in network_collection['graph']:
            G.data = G.data.as_directed()
        self.data = [ 
            DiscreteStateNetwork(np.transpose(G.edges()), y, *args, **kwargs) 
            for (G, y) in zip(network_collection['graph'], labels)
        ]
        self.classes = tuple(classes)
        self.device = device

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx:int):
        return self.data[idx].to(self.device)

    @property
    def labels(self):
        return tc.stack([ G.y for G in self ], 0)

    def _random_loader(self, indices, num_samples:Union[int,float]=0, balance:bool=False, *args, **kwargs):
        subset  = Subset(self, indices)
        if isinstance(num_samples, float):
            num_samples = int(num_samples * len(subset))
        num_samples = min(num_samples, len(subset))
        if not num_samples:
            sampler = None
        elif not balance:
            sampler = RandomSampler(subset, False, num_samples)
        else:
            labels = self.labels[indices, 0]
            chances = 1.0 / np.bincount(labels)
            weights = chances[labels]
            sampler = WeightedRandomSampler(weights, num_samples, True)
        loader = GraphDataLoader(subset, sampler=sampler, *args, **kwargs)
        return loader

    def _index_split(self, pool:list, proportion:list, stratify:bool, seed:int=None):
        indices = []
        taken = 0.0
        for p in proportion:
            percent = p / (1.0 - taken)
            taken += p
            if taken >= 1.0:
                break
            elif percent == 0:
                indices.append([])
                continue
            strat = self.labels[pool].to('cpu') if stratify else None
            selected, pool = train_test_split(pool, train_size=percent, stratify=strat, random_state=seed)
            indices.append(selected)
        indices.append(pool)
        return indices

    def split(self, training:float, validation:float=0, stratify:bool=True, folds:int=0, pool:list=None, sampling:tuple=(0,0,0), 
                repeat:int=0, seed:int=None, *args, **kwargs):
        if 'batch_size' not in kwargs: # temporary, it should be in the named param list
            kwargs['batch_size'] = 1
        if repeat > 0:
            if seed is None:
                seed = [None]*repeat
            return [ 
                self.split(training, validation, stratify, folds, pool, sampling, repeat=0, seed=seed[i], *args, **kwargs) 
                for i in range(repeat) 
            ]
        if pool is None:
            pool = np.arange(len(self))
        if 'batch_size' not in kwargs:
            kwargs['batch_size'] = None
        if folds in [0, 1]:
            (tr_idx, vl_idx, te_idx) = self._index_split(pool, [training, validation], stratify, seed)
            loaders = {
                'train': self._random_loader(tr_idx, sampling[0], *args, **kwargs), 
                'valid': self._random_loader(vl_idx, sampling[1], *args, **kwargs), 
                'test': self._random_loader(te_idx, sampling[-1], *args, **kwargs) 
            }
            return loaders if not folds else [loaders]
        else:
            folds_idx = self._index_split(pool, [1.0/folds] * (folds-1), stratify, seed)
            dataloaders = []
            for te_idx in folds_idx:
                scale = 1 / (training + validation)
                subpool = list(set(pool) - set(te_idx))
                if validation > 0:
                    (tr_idx, vl_idx) = self._index_split(subpool, [training*scale, validation*scale], stratify, seed)
                else:
                    (tr_idx, vl_idx) = (subpool, [])
                dataloaders.append({
                    'train': self._random_loader(tr_idx, sampling[0], *args, **kwargs),
                    'valid': self._random_loader(vl_idx, sampling[1], *args, **kwargs), 
                    'test': self._random_loader(te_idx, sampling[-1], *args, **kwargs)
                })
            return dataloaders

# Load our favourite local update rule

In [ ]:
resolution = 9
beta, sigma = (488, 464)
B_set = binary_indices(beta)
S_set = binary_indices(sigma)

# this rule is equivalent to itself
beta_equiv, sigma_equiv = return_equivalent_rule(resolution, B_set, S_set, return_decimals=True)
if (beta, sigma) == (beta_equiv, sigma_equiv):
    print("This rule is equivalent to itself.")

model = LLNA(resolution, B_set, S_set, iso=True)
model.diagram(degree=8, plot_dist=True)

def init_config_with_dens(N, dens):
    # defines a random initial configuration with a fixed state density
    s0 = np.zeros(N, dtype=int)
    s0[:np.round(dens*N).astype(int)] = 1.
    np.random.shuffle(s0)
    return s0

# Load various graphs from the database

# 1. Synthetic graphs

In [ ]:
#EE import classes from submodule

## fix path
import sys, os
# path to src
sys.path.append(os.path.abspath(".."))
# path to submodule
sys.path.append(os.path.abspath("../src/network_analysis"))

## get classes
from src.network_analysis.lib.datasources import SyntheticNetworks #, KeggMetabolicNetworks

### 1.1 Classic networks

In [ ]:
# load dataframe with networks
base_folder = "..\\src\\network_analysis\\data"
SyntheticNetworks.setup(base_folder=base_folder, random_seed=99999, loader_mode='full')

networks_classic = SyntheticNetworks.classic(samples_per_label=100, balanced=True)

In [ ]:
grouped = networks_classic.groupby('label')
keys = networks_classic.label.unique()
networks_dict = {key: group for key, group in grouped}

dataset_dict = {key: DiscreteStateNetworkDataset(networks_dict[key], num_states=2, num_inits=1) for key in keys}

In [ ]:
RESET=False
if RESET:
    final_config_dict = {}
T=200

for label in keys:
    print(f"Working with label {label} ...")

    final_config_mean_array = []
    dict_length = len(dataset_dict[label])
    for G in tqdm(dataset_dict[label], total=dict_length):
        # load graph and initial configuration  
        (E, h, _, y) = G.unpack()
        edges_classic = E.numpy().T
        num_nodes = h.shape[1]
        # graph_classic = ig.Graph(n=num_nodes, edges=edges_classic)
        # init_config = h # these are not at an average state of exactly 0.5
        init_config = tc.tensor(init_config_with_dens(num_nodes, 0.5)[np.newaxis,:]) # this is a random init config with density 0.5
        configs = model.forward(E, init_config, T=T)
        # find mean of final configuration
        final_config_mean = np.mean(configs[0,-1].numpy())
        final_config_mean_array.append(final_config_mean)
    final_config_mean_array = np.array(final_config_mean_array)
    
    # add to dict
    final_config_dict[label] = final_config_mean_array

In [ ]:
fig, axs = plt.subplots(1,4,figsize=(11,3))
bins = np.linspace(0,1,11)
fontsize=20

for ax, label in zip(axs, keys):
    ax.hist(final_config_dict[label], bins=bins)
    ax.set_title(f"{label}", fontsize=fontsize)
    ax.set_ylim([0,60])

fig.suptitle("Final state means of synthetic classic networks", fontsize=fontsize)

plt.tight_layout()
plt.savefig(f"final-states_rule{model.__str__()}_synthetic-classic-networks.pdf", bbox_inches='tight')

### 1.2 Scalefree networks

In [ ]:
# load dataframe with networks
base_folder = "..\\src\\network_analysis\\data"
SyntheticNetworks.setup(base_folder=base_folder, random_seed=99999, loader_mode='full')

networks_scalefree = SyntheticNetworks.scalefree(samples_per_label=100, balanced=True)

In [ ]:
grouped = networks_scalefree.groupby('label')
keys = networks_scalefree.label.unique()
networks_dict = {key: group for key, group in grouped}

dataset_dict = {key: DiscreteStateNetworkDataset(networks_dict[key], num_states=2, num_inits=1) for key in keys}

Run from initial configuration with exactly the same number of 0s and 1s.

In [ ]:
RESET=False
if RESET:
    final_config_dict = {}
T=200

for label in keys:
    print(f"Working with label {label} ...")

    final_config_mean_array = []
    dict_length = len(dataset_dict[label])
    for G in tqdm(dataset_dict[label], total=dict_length):
        # load graph and initial configuration  
        (E, h, _, y) = G.unpack()
        edges_scalefree = E.numpy().T
        num_nodes = h.shape[1]
        # graph_scalefree = ig.Graph(n=num_nodes, edges=edges_scalefree)
        # init_config = h # these are not at an average state of exactly 0.5
        init_config = tc.tensor(init_config_with_dens(num_nodes, 0.5)[np.newaxis,:]) # this is a random init config with density 0.5
        configs = model.forward(E, init_config, T=T)
        # find mean of final configuration
        final_config_mean = np.mean(configs[0,-1].numpy())
        final_config_mean_array.append(final_config_mean)
    final_config_mean_array = np.array(final_config_mean_array)
    
    # add to dict
    final_config_dict[label] = final_config_mean_array

In [ ]:
fig, axs = plt.subplots(1,5,figsize=(13,3))
bins = np.linspace(0,1,11)
fontsize=20

for ax, label in zip(axs, keys):
    ax.hist(final_config_dict[label], bins=bins)
    ax.set_title(f"{label}", fontsize=fontsize)
    ax.set_ylim([0,60])

fig.suptitle("Final state means of synthetic scalefree networks", fontsize=fontsize)

plt.tight_layout()
plt.savefig(f"final-states_rule{model.__str__()}_synthetic-scalefree-networks.pdf", bbox_inches='tight')

### 1.3 Noise = 10 percent

In [ ]:
# load dataframe with networks
base_folder = "..\\src\\network_analysis\\data"
SyntheticNetworks.setup(base_folder=base_folder, random_seed=99999, loader_mode='full')

networks_noise10 = SyntheticNetworks.noise10(samples_per_label=100, balanced=True)

In [ ]:
grouped = networks_noise10.groupby('label')
keys = networks_noise10.label.unique()
networks_dict = {key: group for key, group in grouped}

dataset_dict = {key: DiscreteStateNetworkDataset(networks_dict[key], num_states=2, num_inits=1) for key in keys}

In [ ]:
RESET=False
if RESET:
    final_config_dict = {}
T=200

for label in keys:
    print(f"Working with label {label} ...")

    final_config_mean_array = []
    dict_length = len(dataset_dict[label])
    for G in tqdm(dataset_dict[label], total=dict_length):
        # load graph and initial configuration  
        (E, h, _, y) = G.unpack()
        edges_noise10 = E.numpy().T
        num_nodes = h.shape[1]
        # graph_noise10 = ig.Graph(n=num_nodes, edges=edges_noise10)
        # init_config = h # these are not at an average state of exactly 0.5
        init_config = tc.tensor(init_config_with_dens(num_nodes, 0.5)[np.newaxis,:]) # this is a random init config with density 0.5
        configs = model.forward(E, init_config, T=T)
        # find mean of final configuration
        final_config_mean = np.mean(configs[0,-1].numpy())
        final_config_mean_array.append(final_config_mean)
    final_config_mean_array = np.array(final_config_mean_array)
    
    # add to dict
    final_config_dict[label] = final_config_mean_array

In [ ]:
fig, axs = plt.subplots(1,8,figsize=(20,3))
bins = np.linspace(0,1,11)
fontsize=20

for ax, label in zip(axs, keys):
    ax.hist(final_config_dict[label], bins=bins)
    ax.set_title(f"{label}", fontsize=fontsize)
    ax.set_ylim([0,60])

fig.suptitle("Final state means of synthetic noise10 networks", fontsize=fontsize)

plt.tight_layout()
plt.savefig(f"final-states_rule{model.__str__()}_synthetic-noise10-networks.pdf", bbox_inches='tight')

# 2. Metabolic networks

In [ ]:
#EE import classes from submodule

## fix path
import sys, os
# path to src
sys.path.append(os.path.abspath(".."))
# path to submodule
sys.path.append(os.path.abspath("../src/network_analysis"))

## get classes
from src.network_analysis.lib.datasources import KeggMetabolicNetworks

### 2.1 Actinobacteria

In [ ]:
# load dataframe with networks
base_folder = "..\\src\\network_analysis\\data"
KeggMetabolicNetworks.setup(base_folder=base_folder, random_seed=99999, loader_mode='full')

networks_actinobac = KeggMetabolicNetworks.actinobac(samples_per_label=100, balanced=True)

In [ ]:
grouped = networks_actinobac.groupby('label')
keys = networks_actinobac.label.unique()
networks_dict = {key: group for key, group in grouped}

dataset_dict = {key: DiscreteStateNetworkDataset(networks_dict[key], num_states=2, num_inits=1) for key in keys}

In [ ]:
RESET=False
if RESET:
    final_config_dict = {}
T=200

for label in keys:
    print(f"Working with label {label} ...")

    final_config_mean_array = []
    dict_length = len(dataset_dict[label])
    for G in tqdm(dataset_dict[label], total=dict_length):
        # load graph and initial configuration  
        (E, h, _, y) = G.unpack()
        edges_actinobac = E.numpy().T
        num_nodes = h.shape[1]
        # graph_actinobac = ig.Graph(n=num_nodes, edges=edges_actinobac)
        # init_config = h # these are not at an average state of exactly 0.5
        init_config = tc.tensor(init_config_with_dens(num_nodes, 0.5)[np.newaxis,:]) # this is a random init config with density 0.5
        configs = model.forward(E, init_config, T=T)
        # find mean of final configuration
        final_config_mean = np.mean(configs[0,-1].numpy())
        final_config_mean_array.append(final_config_mean)
    final_config_mean_array = np.array(final_config_mean_array)
    
    # add to dict
    final_config_dict[label] = final_config_mean_array

In [ ]:
fig, axs = plt.subplots(1,3,figsize=(8,3))
bins = np.linspace(0,1,11)
fontsize=20

for ax, label in zip(axs, keys):
    ax.hist(final_config_dict[label], bins=bins)
    ax.set_title(f"{label}", fontsize=fontsize)
    ax.set_ylim([0,20])

fig.suptitle("Final state means of metabolic actinobacteria networks", fontsize=fontsize)

plt.tight_layout()
plt.savefig(f"final-states_rule{model.__str__()}_metabolic-actinobac-networks.pdf", bbox_inches='tight')

### 2.2 Animals

In [ ]:
# load dataframe with networks
base_folder = "..\\src\\network_analysis\\data"
KeggMetabolicNetworks.setup(base_folder=base_folder, random_seed=99999, loader_mode='full')

networks_animals = KeggMetabolicNetworks.animals(samples_per_label=100, balanced=True)

In [ ]:
grouped = networks_animals.groupby('label')
keys = networks_animals.label.unique()
networks_dict = {key: group for key, group in grouped}

dataset_dict = {key: DiscreteStateNetworkDataset(networks_dict[key], num_states=2, num_inits=1) for key in keys}
print(keys)

In [ ]:
RESET=False
if RESET:
    final_config_dict = {}
T=200

for label in keys:
    print(f"Working with label {label} ...")

    final_config_mean_array = []
    dict_length = len(dataset_dict[label])
    for G in tqdm(dataset_dict[label], total=dict_length):
        # load graph and initial configuration  
        (E, h, _, y) = G.unpack()
        edges_animals = E.numpy().T
        num_nodes = h.shape[1]
        # graph_animals = ig.Graph(n=num_nodes, edges=edges_animals)
        # init_config = h # these are not at an average state of exactly 0.5
        init_config = tc.tensor(init_config_with_dens(num_nodes, 0.5)[np.newaxis,:]) # this is a random init config with density 0.5
        configs = model.forward(E, init_config, T=T)
        # find mean of final configuration
        final_config_mean = np.mean(configs[0,-1].numpy())
        final_config_mean_array.append(final_config_mean)
    final_config_mean_array = np.array(final_config_mean_array)
    
    # add to dict
    final_config_dict[label] = final_config_mean_array

In [ ]:
fig, axs = plt.subplots(1,len(keys),figsize=(8,3))
bins = np.linspace(0,1,11)
fontsize=20

for ax, label in zip(axs, keys):
    ax.hist(final_config_dict[label], bins=bins)
    ax.set_title(f"{label}", fontsize=fontsize)
    ax.set_ylim([0,10])

fig.suptitle("Final state means of metabolic animal networks", fontsize=fontsize)

plt.tight_layout()
plt.savefig(f"final-states_rule{model.__str__()}_metabolic-animal-networks.pdf", bbox_inches='tight')

### 2.3 Firmicutes Bacillis

In [ ]:
# load dataframe with networks
base_folder = "..\\src\\network_analysis\\data"
KeggMetabolicNetworks.setup(base_folder=base_folder, random_seed=99999, loader_mode='full')

networks_fbacillis = KeggMetabolicNetworks.fbacillis(samples_per_label=100, balanced=True)

In [ ]:
grouped = networks_fbacillis.groupby('label')
keys = networks_fbacillis.label.unique()
networks_dict = {key: group for key, group in grouped}

dataset_dict = {key: DiscreteStateNetworkDataset(networks_dict[key], num_states=2, num_inits=1) for key in keys}
print(keys)

In [ ]:
RESET=False
if RESET:
    final_config_dict = {}
T=200

for label in keys:
    print(f"Working with label {label} ...")

    final_config_mean_array = []
    dict_length = len(dataset_dict[label])
    for G in tqdm(dataset_dict[label], total=dict_length):
        # load graph and initial configuration  
        (E, h, _, y) = G.unpack()
        edges_fbacillis = E.numpy().T
        num_nodes = h.shape[1]
        # graph_fbacillis = ig.Graph(n=num_nodes, edges=edges_fbacillis)
        # init_config = h # these are not at an average state of exactly 0.5
        init_config = tc.tensor(init_config_with_dens(num_nodes, 0.5)[np.newaxis,:]) # this is a random init config with density 0.5
        configs = model.forward(E, init_config, T=T)
        # find mean of final configuration
        final_config_mean = np.mean(configs[0,-1].numpy())
        final_config_mean_array.append(final_config_mean)
    final_config_mean_array = np.array(final_config_mean_array)
    
    # add to dict
    final_config_dict[label] = final_config_mean_array

In [ ]:
fig, axs = plt.subplots(1,len(keys),figsize=(8,3))
bins = np.linspace(0,1,11)
fontsize=20

for ax, label in zip(axs, keys):
    ax.hist(final_config_dict[label], bins=bins)
    ax.set_title(f"{label}", fontsize=fontsize)
    ax.set_ylim([0,30])

fig.suptitle("Final state means of metabolic firmicutes bacillis networks", fontsize=fontsize)

plt.tight_layout()
plt.savefig(f"final-states_rule{model.__str__()}_metabolic-fbacillis-networks.pdf", bbox_inches='tight')

### 2.4 Fungi

In [ ]:
# load dataframe with networks
base_folder = "..\\src\\network_analysis\\data"
KeggMetabolicNetworks.setup(base_folder=base_folder, random_seed=99999, loader_mode='full')

networks_fungi = KeggMetabolicNetworks.fungi(samples_per_label=100, balanced=True)

In [ ]:
grouped = networks_fungi.groupby('label')
keys = networks_fungi.label.unique()
networks_dict = {key: group for key, group in grouped}

dataset_dict = {key: DiscreteStateNetworkDataset(networks_dict[key], num_states=2, num_inits=1) for key in keys}
print(keys)

In [ ]:
RESET=False
if RESET:
    final_config_dict = {}
T=200

for label in keys:
    print(f"Working with label {label} ...")

    final_config_mean_array = []
    dict_length = len(dataset_dict[label])
    for G in tqdm(dataset_dict[label], total=dict_length):
        # load graph and initial configuration  
        (E, h, _, y) = G.unpack()
        edges_fungi = E.numpy().T
        num_nodes = h.shape[1]
        # graph_fungi = ig.Graph(n=num_nodes, edges=edges_fungi)
        # init_config = h # these are not at an average state of exactly 0.5
        init_config = tc.tensor(init_config_with_dens(num_nodes, 0.5)[np.newaxis,:]) # this is a random init config with density 0.5
        configs = model.forward(E, init_config, T=T)
        # find mean of final configuration
        final_config_mean = np.mean(configs[0,-1].numpy())
        final_config_mean_array.append(final_config_mean)
    final_config_mean_array = np.array(final_config_mean_array)
    
    # add to dict
    final_config_dict[label] = final_config_mean_array

In [ ]:
fig, axs = plt.subplots(1,len(keys),figsize=(8,3))
bins = np.linspace(0,1,11)
fontsize=20

for ax, label in zip(axs, keys):
    ax.hist(final_config_dict[label], bins=bins)
    ax.set_title(f"{label}", fontsize=fontsize)
    ax.set_ylim([0,10])

fig.suptitle("Final state means of metabolic fungi networks", fontsize=fontsize)

plt.tight_layout()
plt.savefig(f"final-states_rule{model.__str__()}_metabolic-fungi-networks.pdf", bbox_inches='tight')

### 2.5 Kingdom

In [ ]:
# load dataframe with networks
base_folder = "..\\src\\network_analysis\\data"
KeggMetabolicNetworks.setup(base_folder=base_folder, random_seed=99999, loader_mode='full')

networks_kingdom = KeggMetabolicNetworks.kingdom(samples_per_label=100, balanced=True)

In [ ]:
grouped = networks_kingdom.groupby('label')
keys = networks_kingdom.label.unique()
networks_dict = {key: group for key, group in grouped}

dataset_dict = {key: DiscreteStateNetworkDataset(networks_dict[key], num_states=2, num_inits=1) for key in keys}
print(keys)

In [ ]:
RESET=False
if RESET:
    final_config_dict = {}
T=200

for label in keys:
    print(f"Working with label {label} ...")

    final_config_mean_array = []
    dict_length = len(dataset_dict[label])
    for G in tqdm(dataset_dict[label], total=dict_length):
        # load graph and initial configuration  
        (E, h, _, y) = G.unpack()
        edges_kingdom = E.numpy().T
        num_nodes = h.shape[1]
        # graph_kingdom = ig.Graph(n=num_nodes, edges=edges_kingdom)
        # init_config = h # these are not at an average state of exactly 0.5
        init_config = tc.tensor(init_config_with_dens(num_nodes, 0.5)[np.newaxis,:]) # this is a random init config with density 0.5
        configs = model.forward(E, init_config, T=T)
        # find mean of final configuration
        final_config_mean = np.mean(configs[0,-1].numpy())
        final_config_mean_array.append(final_config_mean)
    final_config_mean_array = np.array(final_config_mean_array)
    
    # add to dict
    final_config_dict[label] = final_config_mean_array

In [ ]:
fig, axs = plt.subplots(1,len(keys),figsize=(2*len(keys),3))
bins = np.linspace(0,1,11)
fontsize=20

for ax, label in zip(axs, keys):
    ax.hist(final_config_dict[label], bins=bins)
    ax.set_title(f"{label}", fontsize=fontsize)
    ax.set_ylim([0,30])

fig.suptitle("Final state means of metabolic kingdom networks", fontsize=fontsize)

plt.tight_layout()
plt.savefig(f"final-states_rule{model.__str__()}_metabolic-kingdom-networks.pdf", bbox_inches='tight')

### 2.6 Plant

In [ ]:
# load dataframe with networks
base_folder = "..\\src\\network_analysis\\data"
KeggMetabolicNetworks.setup(base_folder=base_folder, random_seed=99999, loader_mode='full')

networks_plant = KeggMetabolicNetworks.plant(samples_per_label=100, balanced=True)

In [ ]:
grouped = networks_plant.groupby('label')
keys = networks_plant.label.unique()
networks_dict = {key: group for key, group in grouped}

dataset_dict = {key: DiscreteStateNetworkDataset(networks_dict[key], num_states=2, num_inits=1) for key in keys}
print(keys)

In [ ]:
RESET=False
if RESET:
    final_config_dict = {}
T=200

for label in keys:
    print(f"Working with label {label} ...")

    final_config_mean_array = []
    dict_length = len(dataset_dict[label])
    for G in tqdm(dataset_dict[label], total=dict_length):
        # load graph and initial configuration  
        (E, h, _, y) = G.unpack()
        edges_plant = E.numpy().T
        num_nodes = h.shape[1]
        # graph_plant = ig.Graph(n=num_nodes, edges=edges_plant)
        # init_config = h # these are not at an average state of exactly 0.5
        init_config = tc.tensor(init_config_with_dens(num_nodes, 0.5)[np.newaxis,:]) # this is a random init config with density 0.5
        configs = model.forward(E, init_config, T=T)
        # find mean of final configuration
        final_config_mean = np.mean(configs[0,-1].numpy())
        final_config_mean_array.append(final_config_mean)
    final_config_mean_array = np.array(final_config_mean_array)
    
    # add to dict
    final_config_dict[label] = final_config_mean_array

In [ ]:
fig, axs = plt.subplots(1,len(keys),figsize=(2*len(keys),3))
bins = np.linspace(0,1,11)
fontsize=20

for ax, label in zip(axs, keys):
    ax.hist(final_config_dict[label], bins=bins)
    ax.set_title(f"{label}", fontsize=fontsize)
    ax.set_ylim([0,10])

fig.suptitle("Final state means of metabolic plant networks", fontsize=fontsize)

plt.tight_layout()
plt.savefig(f"final-states_rule{model.__str__()}_metabolic-plant-networks.pdf", bbox_inches='tight')

### 2.7 Protist

In [ ]:
# load dataframe with networks
base_folder = "..\\src\\network_analysis\\data"
KeggMetabolicNetworks.setup(base_folder=base_folder, random_seed=99999, loader_mode='full')

networks_protist = KeggMetabolicNetworks.protist(samples_per_label=100, balanced=True)

In [ ]:
grouped = networks_protist.groupby('label')
keys = networks_protist.label.unique()
networks_dict = {key: group for key, group in grouped}

dataset_dict = {key: DiscreteStateNetworkDataset(networks_dict[key], num_states=2, num_inits=1) for key in keys}
print(keys)

In [ ]:
RESET=False
if RESET:
    final_config_dict = {}
T=200

for label in keys:
    print(f"Working with label {label} ...")

    final_config_mean_array = []
    dict_length = len(dataset_dict[label])
    for G in tqdm(dataset_dict[label], total=dict_length):
        # load graph and initial configuration  
        (E, h, _, y) = G.unpack()
        edges_protist = E.numpy().T
        num_nodes = h.shape[1]
        # graph_protist = ig.Graph(n=num_nodes, edges=edges_protist)
        # init_config = h # these are not at an average state of exactly 0.5
        init_config = tc.tensor(init_config_with_dens(num_nodes, 0.5)[np.newaxis,:]) # this is a random init config with density 0.5
        configs = model.forward(E, init_config, T=T)
        # find mean of final configuration
        final_config_mean = np.mean(configs[0,-1].numpy())
        final_config_mean_array.append(final_config_mean)
    final_config_mean_array = np.array(final_config_mean_array)
    
    # add to dict
    final_config_dict[label] = final_config_mean_array

In [ ]:
fig, axs = plt.subplots(1,len(keys),figsize=(2*len(keys),3))
bins = np.linspace(0,1,11)
fontsize=20

for ax, label in zip(axs, keys):
    ax.hist(final_config_dict[label], bins=bins)
    ax.set_title(f"{label}", fontsize=fontsize)
    ax.set_ylim([0,10])

fig.suptitle("Final state means of metabolic protist networks", fontsize=fontsize)

plt.tight_layout()
plt.savefig(f"final-states_rule{model.__str__()}_metabolic-protist-networks.pdf", bbox_inches='tight')